# 02 Music-Based Intervention Design Extraction

In [ ]:
import json
import re
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI
from pypdf import PdfReader


OPENAI_BASE_URL = ""
OPENAI_API_KEY = ""
MODEL_NAME = ""

EXCEL_PATH = r""
OUTPUT_EXCEL_PATH = r""
PDF_DIR = r""
SHEET_NAME = ""

temperature = 0.2
max_tokens = 4096
RETRY_LIMIT = 5
SLEEP_SECONDS = 10

TARGET_COLUMNS = [
    "InterventionType",
    "Intervention Setting",
    "Intervention Device",
    "Intervention Design",
    "Is Combination Therapy",
    "Combination Therapy Type",
    "Combination Therapy Detail",
]

PROMPT_TEMPLATE = """
You are extracting structured information from a music-therapy-related paper.

Your task is to extract ONLY the following fields from the original paper:

InterventionType:
Definition: The mode of music intervention used in the study.
Extraction instruction: Extract what participants received or did as the music-based intervention, such as listening, singing, instrument playing, movement, composition, improvisation, music training, or a combined approach.

Intervention Setting:
Definition: The location or environment where the intervention occurs.
Extraction instruction: Extract the reported setting where the music intervention was delivered, such as hospital ward, rehabilitation center, nursing home, home-based setting, clinic, community setting, or online setting.

Intervention Device:
Definition: Equipment used in delivering the music intervention.
Extraction instruction: Extract devices or equipment used to deliver or support the music intervention, such as headphones, speakers, mobile apps, live instruments, software, robots, or other devices.

Intervention Design:
Definition: The structured design or protocol of the music intervention.
Extraction instruction: Extract a concise description of how the intervention was organized or delivered, including the main activity, individual or group format, and intervention structure when reported.

Is Combination Therapy:
Definition: Whether the music intervention is delivered in conjunction with other treatments.
Extraction instruction: Extract whether the music intervention was combined with another treatment or care component, such as medication, surgery, chemotherapy, psychotherapy, rehabilitation, standard care, or other non-music therapy.

Combination Therapy Type:
Definition: The type of co-delivered therapy used together with the music intervention.
Extraction instruction: Extract the broad type of non-music co-intervention reported in the paper.

Combination Therapy Detail:
Definition: Specific details of the co-intervention.
Extraction instruction: Extract drug names, procedure names, therapy protocols, standard-care descriptions, or other specific co-intervention details when reported.

Paper metadata:
PMID: {pmid}
Title: {title}
Abstract: {abstract}

Paper text:
{paper_text}

Notes:
1. If the article does not report a field, do not infer or guess; output N/R.
2. Use only information explicitly stated in the paper.
3. Do not use outside knowledge.
4. Output concise English values. For multiple values, use ; as the separator.
5. Output JSON only. No explanation, no reasoning, no markdown.

Return JSON with exactly these keys:
{json_schema}
"""


def require_filled(value: str, name: str) -> str:
    if not str(value).strip():
        raise ValueError(f"Please fill {name} before running this notebook.")
    return str(value).strip()


def normalize_pmid(value) -> str:
    if pd.isna(value):
        return ""
    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()


def read_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    texts = []
    for page in reader.pages:
        text = page.extract_text() or ""
        if text.strip():
            texts.append(text)
    return "\n".join(texts).strip()


def clean_response_to_json(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    return match.group(0) if match else text


def parse_model_output(text: str) -> dict:
    data = json.loads(clean_response_to_json(text))
    return {col: str(data.get(col, "N/R") or "N/R").strip() for col in TARGET_COLUMNS}


base_url = OPENAI_BASE_URL.strip()
api_key = require_filled(OPENAI_API_KEY, "OPENAI_API_KEY")
model = require_filled(MODEL_NAME, "MODEL_NAME")
excel_path = Path(require_filled(EXCEL_PATH, "EXCEL_PATH"))
output_excel_path = Path(require_filled(OUTPUT_EXCEL_PATH, "OUTPUT_EXCEL_PATH"))
pdf_dir = Path(require_filled(PDF_DIR, "PDF_DIR"))

client_kwargs = {"api_key": api_key}
if base_url:
    client_kwargs["base_url"] = base_url
client = OpenAI(**client_kwargs)

sheet = SHEET_NAME.strip() or 0
df = pd.read_excel(excel_path, sheet_name=sheet)
for col in TARGET_COLUMNS:
    if col not in df.columns:
        df[col] = ""

json_schema = json.dumps({col: "" for col in TARGET_COLUMNS}, indent=2, ensure_ascii=False)

for idx, row in df.iterrows():
    pmid = normalize_pmid(row.get("PMID", ""))
    if not pmid:
        continue

    pdf_path = pdf_dir / f"{pmid}.pdf"
    title = "" if pd.isna(row.get("Title", "")) else str(row.get("Title", ""))
    abstract = "" if pd.isna(row.get("Abstract", "")) else str(row.get("Abstract", ""))

    if not pdf_path.exists():
        print(f"Skip PMID {pmid}: PDF not found")
        continue

    try:
        paper_text = read_pdf_text(pdf_path)
    except Exception as e:
        print(f"Skip PMID {pmid}: PDF read error: {e}")
        continue

    if not paper_text:
        paper_text = abstract

    prompt = PROMPT_TEMPLATE.format(pmid=pmid, title=title, abstract=abstract, paper_text=paper_text, json_schema=json_schema)

    success = False
    for attempt in range(RETRY_LIMIT):
        try:
            start_time = time.perf_counter()
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a careful medical literature information extraction assistant."},
                    {"role": "user", "content": prompt},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            answer = response.choices[0].message.content.strip()
            parsed = parse_model_output(answer)
            for col, value in parsed.items():
                df.at[idx, col] = value if value else "N/R"
            elapsed = round(time.perf_counter() - start_time, 3)
            print(f"Row {idx + 2} PMID {pmid} done in {elapsed}s")
            output_excel_path.parent.mkdir(parents=True, exist_ok=True)
            df.to_excel(output_excel_path, index=False)
            success = True
            break
        except Exception as e:
            print(f"Error processing row {idx + 2} PMID {pmid}, attempt {attempt + 1}: {e}")
            if attempt < RETRY_LIMIT - 1:
                time.sleep(SLEEP_SECONDS)

    if not success:
        print(f"Failed PMID {pmid}")

output_excel_path.parent.mkdir(parents=True, exist_ok=True)
df.to_excel(output_excel_path, index=False)
print(f"All done! Results saved to {output_excel_path}")
